# Real-fMRI validation (LeBel et al. 2023), text + audio

Runs TRIBE v2 on six stories of subject UTS03 with **text (Llama-3.2-3B) + audio (w2v-BERT 2.0)** features, then tests encoder inversion on a held-out story. Reproduces Table VI of the manuscript and `results/realdata_audio_text.json`.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. You need a Hugging Face token with access to `meta-llama/Llama-3.2-3B`.

Run the cells **one at a time, top to bottom**. Data and features are saved to Google Drive (`MyDrive/tribe_inversion`), so a disconnect loses nothing: re-run Step 2 and Step 4, then continue.

Total time on a free T4: about 1.5–2 h (mostly Step 6).

## Step 1 · Install and clone the repository (≈3 min)
When it finishes, click **Runtime → Restart session**, then go to Step 2. Do not run Step 1 again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install uv
!uv pip -q install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip -q install --system "transformers<4.57" "torchaudio==2.6.0" "numpy<2.1" h5py scikit-learn
!git clone -q https://github.com/Ivxn404/semantic-decoding-tribe-inversion /content/repo || (cd /content/repo && git pull -q)
print('INSTALL DONE - now click Runtime > Restart session, then continue with the next step')

## Step 2 · Settings and Google Drive (run again after any disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
os.environ['PROJECT_ROOT'] = '/content/repo'
os.environ['LEBEL_WORK'] = '/content/drive/MyDrive/tribe_inversion/lebel'   # data + features + results
os.environ['TRIBE_WEIGHTS'] = '/content/repo/weights'
os.environ['ATLAS_DIR'] = '/content/atlas'
os.makedirs(os.environ['LEBEL_WORK'] + '/pycx', exist_ok=True)
!cp -n /content/repo/data/lebel/pycx/sampler.npz "$LEBEL_WORK/pycx/"
os.chdir('/content/repo/scripts')
sys.path = [p for p in sys.path if not p.endswith('tribev2')]
import torch; print('GPU:', torch.cuda.is_available())

## Step 3 · Log in to Hugging Face
Paste your token in the box that appears. It stays in your Colab session and is not saved in the notebook.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 4 · Download the TRIBE v2 weights (≈1 min; re-run after a disconnect)

In [ ]:
from huggingface_hub import hf_hub_download
for f in ['config.yaml', 'best.ckpt']:
    hf_hub_download('facebook/tribev2', f, local_dir='/content/repo/weights')
!ls -la /content/repo/weights

## Step 5 · Download the fMRI and map it onto the cortical surface (≈5–10 min)
Downloads UTS03's preprocessed responses from OpenNeuro (ds003020) and projects them to fsaverage5 with the validated sampling matrix in `data/lebel/pycx/sampler.npz`.

In [ ]:
!python lebel_apply.py
!ls "$LEBEL_WORK/bold_fs5"

## Step 6 · Stimulus features, one story per run (≈40–60 min)
Each story runs in a fresh process with audio first and text second, so only one large model is on the 15-GB GPU at a time. Llama-3.2-3B is loaded in half precision on GPUs under 20 GB. Finished stories are skipped if you re-run.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for st in ['wheretheressmoke', 'adollshouse', 'avatar', 'legacy', 'souls', 'undertheinfluence']:
    print('=====', st)
    !python lebel_features.py --features audio --chunk 60 --weights /content/repo/weights --stories $st 2>&1 | tail -2
    !python lebel_features.py --features audio text --chunk 60 --weights /content/repo/weights --stories $st 2>&1 | tail -3
!ls "$LEBEL_WORK/feats_audio_text"   # expect six .npz files

## Step 7 · Evaluate (≈10–20 min)
Alignment, encoding accuracy per network with a noise ceiling, forward identification, and inversion of measured BOLD with an oracle control.

In [ ]:
!python lebel_eval.py --tag audio_text --steps 2000 --weights /content/repo/weights 2>&1 | grep -iv warn | tail -15

## Step 8 · Results

In [ ]:
import json, os
r = json.load(open(os.path.join(os.environ['LEBEL_WORK'], 'realdata_audio_text.json')))
print('lag (s):', r['lag']['best_s'])
print('encoding r, held-out:', {k: round(v, 3) for k, v in r['encoding_r']['wheretheressmoke'].items()})
print('noise ceiling:', r['noise_ceiling'])
print('forward identification (top-2000):', r['identification_forward']['top2000'])
print('inversion, real BOLD:', {k: v for k, v in r['inversion_identification']['real_bold'].items() if k != 'ranks'})
print('inversion, oracle:', {k: v for k, v in r['inversion_identification']['oracle_prediction'].items() if k != 'ranks'})